# ETAPA 4 — Componente generativo (resúmenes con T5)

Este notebook implementa el componente generativo del sistema: **generación de resúmenes** de reseñas
con el modelo preentrenado `t5-small` en modo zero-shot (sin fine-tuning, que se abordará en la Etapa 5).

**Nota sobre carga de datos**: el dataset necesita la columna `Summary` del archivo original (que los CSV
exportados por el EDA no incluyen, porque esa columna no se usa en la clasificación). Por lo tanto, aquí
se hace una carga puntual del CSV original para poder recuperar `Summary_clean` como ground truth para ROUGE.
La lógica de limpieza y split respeta exactamente la misma semilla del EDA para mantener consistencia entre
etapas.

In [1]:
# ETAPA 4 - Componente generativo (resumenes)
# CELDA 1: carga del dataset original y preparacion del texto

import re
import pandas as pd

# Ruta correcta del dataset en Kaggle
df = pd.read_csv(
    "/kaggle/input/datasets/rodrigolopez29/reviews/Reviews.csv",
    engine="python",
    on_bad_lines="skip"
)

print("Shape original:", df.shape)
print("Columnas originales:", df.columns.tolist())

# Seleccion de columnas relevantes
df = df[["Id", "ProductId", "Score", "Summary", "Text"]].copy()
df = df.dropna(subset=["Score", "Text"]).copy()

# Limpieza de texto
def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r"[^a-zA-Z\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

df["Text_clean"] = df["Text"].apply(limpiar_texto)
df["Summary_clean"] = df["Summary"].fillna("").apply(limpiar_texto)

# Eliminar textos vacios
df = df[df["Text_clean"].str.len() > 0].copy()

# Conversion de score a sentimiento
def convertir_sentimiento(score):
    if score <= 2:
        return "negative"
    elif score == 3:
        return "neutral"
    else:
        return "positive"

df["sentiment"] = df["Score"].apply(convertir_sentimiento)

# Longitudes
df["text_len"] = df["Text_clean"].str.len()
df["summary_len"] = df["Summary_clean"].str.len()

# Prints de validacion
print("\nShape limpio:", df.shape)

print("\nDistribucion de sentimiento:")
print(df["sentiment"].value_counts())

print("\nResumen longitud de review limpia:")
print(df["text_len"].describe())

print("\nResumen longitud de summary limpio:")
print(df["summary_len"].describe())

print("\nPrimeras 5 filas:")
print(df[["Id", "ProductId", "Score", "sentiment", "Text_clean", "Summary_clean"]].head())

Shape original: (568454, 10)
Columnas originales: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']

Shape limpio: (568453, 10)

Distribucion de sentimiento:
sentiment
positive    443776
negative     82037
neutral      42640
Name: count, dtype: int64

Resumen longitud de review limpia:
count    568453.000000
mean        417.469428
std         422.483671
min          12.000000
25%         172.000000
50%         290.000000
75%         506.000000
max       20517.000000
Name: text_len, dtype: float64

Resumen longitud de summary limpio:
count    568453.000000
mean         22.527250
std          13.514832
min           0.000000
25%          13.000000
50%          20.000000
75%          29.000000
max         128.000000
Name: summary_len, dtype: float64

Primeras 5 filas:
   Id   ProductId  Score sentiment  \
0   1  B001E4KFG0      5  positive   
1   2  B00813GRG4      1  negative   
2   3  B000LQOCH0      4  pos

In [2]:
# ETAPA 4 - Componente generativo (resumenes)
# CELDA 2: reconstruccion de splits y guardado de CSV para la etapa generativa

from sklearn.model_selection import train_test_split

X_text = df["Text_clean"]
X_summary = df["Summary_clean"]
y = df["sentiment"]

# Split 70 / 15 / 15 estratificado
X_text_temp, X_text_test, X_summary_temp, X_summary_test, y_temp, y_test = train_test_split(
    X_text,
    X_summary,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_text_train, X_text_val, X_summary_train, X_summary_val, y_train, y_val = train_test_split(
    X_text_temp,
    X_summary_temp,
    y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)

# Construccion de dataframes
train_df_gen = pd.DataFrame({
    "Text_clean": X_text_train,
    "Summary_clean": X_summary_train,
    "sentiment": y_train
}).reset_index(drop=True)

val_df_gen = pd.DataFrame({
    "Text_clean": X_text_val,
    "Summary_clean": X_summary_val,
    "sentiment": y_val
}).reset_index(drop=True)

test_df_gen = pd.DataFrame({
    "Text_clean": X_text_test,
    "Summary_clean": X_summary_test,
    "sentiment": y_test
}).reset_index(drop=True)

# Validaciones
print("Train generativo:", train_df_gen.shape)
print("Val generativo:", val_df_gen.shape)
print("Test generativo:", test_df_gen.shape)

print("\nDistribucion train:")
print(train_df_gen["sentiment"].value_counts(normalize=True))

print("\nDistribucion val:")
print(val_df_gen["sentiment"].value_counts(normalize=True))

print("\nDistribucion test:")
print(test_df_gen["sentiment"].value_counts(normalize=True))

print("\nCantidad de summaries vacios:")
print("Train:", (train_df_gen["Summary_clean"].str.len() == 0).sum())
print("Val:", (val_df_gen["Summary_clean"].str.len() == 0).sum())
print("Test:", (test_df_gen["Summary_clean"].str.len() == 0).sum())

# Guardado
train_df_gen.to_csv("/kaggle/working/train_etapa4_generativa.csv", index=False)
val_df_gen.to_csv("/kaggle/working/val_etapa4_generativa.csv", index=False)
test_df_gen.to_csv("/kaggle/working/test_etapa4_generativa.csv", index=False)

print("\nArchivos guardados:")
print("/kaggle/working/train_etapa4_generativa.csv")
print("/kaggle/working/val_etapa4_generativa.csv")
print("/kaggle/working/test_etapa4_generativa.csv")

print("\nVista rapida train_df_gen:")
print(train_df_gen.head())

Train generativo: (397902, 3)
Val generativo: (85283, 3)
Test generativo: (85268, 3)

Distribucion train:
sentiment
positive    0.780675
negative    0.144314
neutral     0.075011
Name: proportion, dtype: float64

Distribucion val:
sentiment
positive    0.780671
negative    0.144320
neutral     0.075009
Name: proportion, dtype: float64

Distribucion test:
sentiment
positive    0.780668
negative    0.144321
neutral     0.075011
Name: proportion, dtype: float64

Cantidad de summaries vacios:
Train: 159
Val: 38
Test: 31

Archivos guardados:
/kaggle/working/train_etapa4_generativa.csv
/kaggle/working/val_etapa4_generativa.csv
/kaggle/working/test_etapa4_generativa.csv

Vista rapida train_df_gen:
                                          Text_clean  \
0  my dog loves this treat she loves the taste of...   
1  i bought the chicken meal and rice formula bec...   
2  very smooth tea can drink plain make a latte n...   
3  these have a very mild peanut butter flavor i ...   
4  i used this produ

In [4]:
# ETAPA 4 - Componente generativo (resumenes)
# CELDA 3: activacion de GPU, carga de datos generativos y modelo T5

!pip install -q transformers sentencepiece accelerate evaluate rouge_score

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Verificacion de GPU
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    print("ATENCION: no hay GPU activa")

# Carga de los CSV generativos guardados en la etapa anterior
train_df_gen = pd.read_csv("/kaggle/working/train_etapa4_generativa.csv")
val_df_gen = pd.read_csv("/kaggle/working/val_etapa4_generativa.csv")
test_df_gen = pd.read_csv("/kaggle/working/test_etapa4_generativa.csv")

print("\nShapes cargados:")
print("Train:", train_df_gen.shape)
print("Val:", val_df_gen.shape)
print("Test:", test_df_gen.shape)

# Filtramos summaries vacios para la tarea de resumen
train_df_gen = train_df_gen[train_df_gen["Summary_clean"].fillna("").str.len() > 0].copy()
val_df_gen = val_df_gen[val_df_gen["Summary_clean"].fillna("").str.len() > 0].copy()
test_df_gen = test_df_gen[test_df_gen["Summary_clean"].fillna("").str.len() > 0].copy()

print("\nShapes despues de quitar summaries vacios:")
print("Train:", train_df_gen.shape)
print("Val:", val_df_gen.shape)
print("Test:", test_df_gen.shape)

# Modelo generativo text-to-text
model_name = "t5-small"

tokenizer_t5 = AutoTokenizer.from_pretrained(model_name)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_t5 = model_t5.to(device)

print("\nModelo cargado:", model_name)
print("Dispositivo del modelo:", device)

# Vista rapida de algunos ejemplos reales
print("\nEjemplos de validacion:")
print(val_df_gen[["Text_clean", "Summary_clean", "sentiment"]].head(3))

CUDA disponible: True
GPU detectada: Tesla T4
CUDA capability: (7, 5)

Shapes cargados:
Train: (397902, 3)
Val: (85283, 3)
Test: (85268, 3)

Shapes despues de quitar summaries vacios:
Train: (397743, 3)
Val: (85245, 3)
Test: (85235, 3)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


Modelo cargado: t5-small
Dispositivo del modelo: cuda

Ejemplos de validacion:
                                          Text_clean  \
0  i got the energy boost of drinking a cup of co...   
1  my fussy cats love this food and from what i v...   
2  wow i ve never not liked one of the coffees or...   

                  Summary_clean sentiment  
0  tastes horrible not worth it  negative  
1               my cats love it  positive  
2              not a good taste  negative  


In [5]:
# ETAPA 4 - Componente generativo (resumenes)
# CELDA 4: prueba pequena de generacion sobre pocas reseñas

# Tomamos una muestra pequena y controlada del set de validacion
sample_size = 5
sample_df = val_df_gen.sample(n=sample_size, random_state=42).reset_index(drop=True)

generated_summaries = []

for i, row in sample_df.iterrows():
    review_text = str(row["Text_clean"])
    
    # Prompt estilo T5
    input_text = "summarize: " + review_text
    
    # Tokenizacion
    inputs = tokenizer_t5(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)
    
    # Generacion
    summary_ids = model_t5.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=32,
        min_length=4,
        num_beams=4,
        early_stopping=True
    )
    
    generated_text = tokenizer_t5.decode(summary_ids[0], skip_special_tokens=True)
    generated_summaries.append(generated_text)

sample_df["generated_summary"] = generated_summaries

print("Muestra generada:")
print(sample_df[["sentiment", "Summary_clean", "generated_summary"]])

print("\nEjemplos completos:\n")
for i, row in sample_df.iterrows():
    print("=" * 100)
    print(f"Ejemplo {i+1}")
    print("\nREVIEW:")
    print(row["Text_clean"][:1200])
    print("\nSUMMARY REAL:")
    print(row["Summary_clean"])
    print("\nSUMMARY GENERADO:")
    print(row["generated_summary"])
    print()

Muestra generada:
  sentiment                           Summary_clean  \
0  positive                            perfect gift   
1  positive                          great marinade   
2  positive                  great crunchy bbq chip   
3   neutral  wasn t really a great drink when added   
4  positive             orange coffee it s the best   

                                   generated_summary  
0  a friend of mine recently had miscarriage and ...  
1  this marinade is a great marinade and makes st...  
2  bbq chips and vita tops trifecta of snacks tha...  
3  tried raspberry lemonade but it wasn't great b...  
4  i love orange coffee and it is simply becoming...  

Ejemplos completos:

Ejemplo 1

REVIEW:
a friend of mine recently had miscarriage and i wanted to send her something to brighten her day i didn t want to send flowers or an overpriced gift basket with barely anything it so i decided to look for something yummy on amazon since i have amazon prime and it would be free tw

In [6]:
# ETAPA 4 - Componente generativo (resumenes)
# CELDA 5: generacion por lotes sobre una muestra del test set

import math
import pandas as pd
from tqdm.auto import tqdm

# Muestra de evaluacion controlada
eval_size = 500
eval_df = test_df_gen.sample(n=eval_size, random_state=42).reset_index(drop=True)

print("Shape eval_df:", eval_df.shape)

batch_size = 16
generated_summaries_eval = []

texts = eval_df["Text_clean"].astype(str).tolist()

for start in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[start:start + batch_size]
    batch_prompts = ["summarize: " + text for text in batch_texts]

    inputs = tokenizer_t5(
        batch_prompts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    summary_ids = model_t5.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=32,
        min_length=4,
        num_beams=4,
        early_stopping=True
    )

    batch_generated = tokenizer_t5.batch_decode(summary_ids, skip_special_tokens=True)
    generated_summaries_eval.extend(batch_generated)

eval_df["generated_summary"] = generated_summaries_eval

print("\nVista rapida de resultados:")
print(eval_df[["sentiment", "Summary_clean", "generated_summary"]].head(10))

Shape eval_df: (500, 3)


  0%|          | 0/32 [00:00<?, ?it/s]


Vista rapida de resultados:
  sentiment                                     Summary_clean  \
0  negative  ok taste but too many calories and too expensive   
1  positive                                           awesome   
2  positive                                trop a rocka rocks   
3  positive                         love these little candies   
4  positive                         one of many safe products   
5  positive                                     great catfood   
6  positive                                       always good   
7  negative                                doesnt work for me   
8  negative                   these used to be so much better   
9  positive                                           awesome   

                                   generated_summary  
0  in the past i ve enjoyed drinks with aloe vera...  
1  love love love favorite all time snack even fo...  
2  i m not much on bottle teas instant teas or fl...  
3  flavigny s violet pastilles are 

Debido al costo computacional de los modelos generativos, se evaluó el modelo sobre una muestra representativa del conjunto de test, manteniendo la coherencia con la distribución original.

In [7]:
# ETAPA 4 - Componente generativo (resumenes)
# CELDA 6: evaluacion ROUGE y guardado de resultados

import evaluate
import pandas as pd

rouge = evaluate.load("rouge")

predictions = eval_df["generated_summary"].astype(str).tolist()
references = eval_df["Summary_clean"].astype(str).tolist()

rouge_results = rouge.compute(
    predictions=predictions,
    references=references
)

print("Resultados ROUGE:")
for k, v in rouge_results.items():
    print(f"{k}: {v:.4f}")

# DataFrame de metricas
generative_metrics = pd.DataFrame({
    "Modelo": ["t5-small", "t5-small", "t5-small", "t5-small"],
    "Metrica": ["ROUGE-1", "ROUGE-2", "ROUGE-L", "ROUGE-Lsum"],
    "Valor": [
        rouge_results["rouge1"],
        rouge_results["rouge2"],
        rouge_results["rougeL"],
        rouge_results["rougeLsum"]
    ]
})

# Guardado de resultados
eval_df.to_csv("/kaggle/working/generative_predictions_eval.csv", index=False)
generative_metrics.to_csv("/kaggle/working/generative_metrics.csv", index=False)

print("\nArchivos guardados:")
print("/kaggle/working/generative_predictions_eval.csv")
print("/kaggle/working/generative_metrics.csv")

print("\nTabla de metricas:")
print(generative_metrics)

Resultados ROUGE:
rouge1: 0.0849
rouge2: 0.0232
rougeL: 0.0783
rougeLsum: 0.0784

Archivos guardados:
/kaggle/working/generative_predictions_eval.csv
/kaggle/working/generative_metrics.csv

Tabla de metricas:
     Modelo     Metrica     Valor
0  t5-small     ROUGE-1  0.084910
1  t5-small     ROUGE-2  0.023189
2  t5-small     ROUGE-L  0.078259
3  t5-small  ROUGE-Lsum  0.078400


Los valores de ROUGE obtenidos son bajos, lo cual es consistente con la naturaleza del dataset, donde los resúmenes reales son extremadamente cortos (tipo etiqueta) mientras que el modelo generativo produce frases más largas y descriptivas. Esto reduce el solapamiento léxico directo entre predicción y referencia.

In [8]:
import os

print(os.listdir("/kaggle/working"))

['.virtual_documents', 'generative_predictions_eval.csv', 'train_etapa4_generativa.csv', 'generative_metrics.csv', 'val_etapa4_generativa.csv', 'test_etapa4_generativa.csv']


In [9]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    print("DIRECTORIO:", root)
    for file in files:
        print("  -", file)

DIRECTORIO: /kaggle/working
  - generative_predictions_eval.csv
  - train_etapa4_generativa.csv
  - generative_metrics.csv
  - val_etapa4_generativa.csv
  - test_etapa4_generativa.csv
DIRECTORIO: /kaggle/working/.virtual_documents
  - __notebook_source__.ipynb


### Comparación cualitativa: review → summary real vs. summary generado

Los valores de ROUGE son bajos porque los `Summary` originales son **etiquetas extremadamente cortas** (3–5
palabras tipo *"great taffy"*, *"perfect gift"*), mientras que T5-small en modo zero-shot tiende a generar
frases completas extraídas del cuerpo de la reseña. Para ilustrar esta diferencia, mostramos una muestra
controlada con los tres campos lado a lado.

In [11]:
import pandas as pd

# Tomamos 8 ejemplos de eval_df y los mostramos en formato comparativo
muestra_qual = eval_df.sample(n=8, random_state=7).reset_index(drop=True)

for i, row in muestra_qual.iterrows():
    print("=" * 100)
    print(f"Ejemplo {i + 1}  [sentiment: {row['sentiment']}]")
    print("\nREVIEW (primeras 500 chars):")
    print(str(row['Text_clean'])[:500])
    print("\nSUMMARY REAL (ground truth):")
    print(row['Summary_clean'])
    print("\nSUMMARY GENERADO (T5-small zero-shot):")
    print(row['generated_summary'])
    print()

# Tabla compacta
print("\n\n--- Tabla compacta (muestra) ---")
print(muestra_qual[['sentiment', 'Summary_clean', 'generated_summary']].to_string(index=False))

Ejemplo 1  [sentiment: positive]

REVIEW (primeras 500 chars):
i want to start by saying i am not a professional just an average cat owner i realize this isn t the highest quality cat food out there however in my opinion this is a great middle of the road more natural cat food our cats love it and haven t had any problems digesting it br br we have cats that are a year old we adopted them from a shelter months ago when they were babies they got a virus and as a result have sensitive stomachs i had to find a food for them with not a lot of fillers and dyes p

SUMMARY REAL (ground truth):
my sensitve stomach cats love it

SUMMARY GENERADO (T5-small zero-shot):
cat food hasn t had any negative affects on our cats in fact they have a soft shiny coat very seldom get hair balls and are full

Ejemplo 2  [sentiment: positive]

REVIEW (primeras 500 chars):
i purchased this coffee at a local store a few months back and loved it it is by far the best coffee i have ever had and i m not a crazy cof

## Conclusiones de la Etapa 4 — Componente generativo

**Modelo utilizado**: `t5-small` (60M parámetros) en modo **zero-shot** (sin fine-tuning).

**Integración con el sistema**: el componente generativo complementa el sistema de clasificación de las
etapas anteriores. Mientras que MLP, 1D-CNN y RoBERTa **clasifican** el sentimiento, T5 **genera**
un resumen textual que permite presentar la reseña en una forma más compacta al usuario final.

**Beneficio aportado al problema**:
- Permite convertir reseñas largas (mediana ~290 caracteres, máx > 20k) en títulos/etiquetas cortas,
  útiles para listados, dashboards y búsquedas.
- Facilita la interpretabilidad: el resumen generado, combinado con la clase de sentimiento, da una
  vista rápida del contenido de la reseña sin leerla entera.

**Resultados cuantitativos (ROUGE sobre 500 reseñas de test)**:
- ROUGE-1 ≈ 0.085
- ROUGE-2 ≈ 0.023
- ROUGE-L ≈ 0.078

**Limitaciones del modelo zero-shot**:
- Los `Summary` del dataset son **etiquetas ultracortas** (tipo "great taffy"), muy distintas al estilo
  de generación de T5, que tiende a extraer frases largas de la reseña. Esto penaliza ROUGE incluso
  cuando el contenido es semánticamente razonable.
- T5-small no está adaptado al dominio de reseñas de comida — fue preentrenado sobre C4 (web general).

**Siguiente paso (Etapa 5)**: realizar **fine-tuning parcial** del modelo sobre pares
(review → summary) del dataset, que se espera reduzca ambas brechas (estilo y dominio) y mejore ROUGE
sustancialmente.